# Scaling laws: finish missing AG News / SST-2 coreset points

Runs only missing `random` and `k_centers` results for `ag_news` and `sst2` across the full learner list. Existing points from `results/scaling_laws/scaling_laws_all_methods.csv` and the aggregation missing-grid are respected, so already available `bert-base-uncased` / `roberta-base` points are not rerun.

In [ ]:
import gc
import json
import os
import runpy
import sys
import time
import traceback
from pathlib import Path

# Limit BLAS/OMP threads before heavy imports.
for _v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "32")

cwd = Path.cwd().resolve()
if (cwd / "notebooks" / "dilm_wrapper.py").exists():
    ROOT = cwd
    NOTEBOOK_DIR = cwd / "notebooks"
elif (cwd / "dilm_wrapper.py").exists():
    NOTEBOOK_DIR = cwd
    ROOT = cwd.parent
else:
    raise RuntimeError(f"Cannot locate project root from {cwd}")

_HF_CACHE = ROOT / "hf_cache"
if _HF_CACHE.exists():
    os.environ.setdefault("HF_HOME", str(_HF_CACHE))
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
    os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
    print(f"Offline mode: HF_HOME={_HF_CACHE}")

sys.path.insert(0, str(NOTEBOOK_DIR))
sys.path.insert(0, str(ROOT / "src"))

import mlflow
import pandas as pd
import torch
from transformers import set_seed

from dilm_wrapper import (
    RESULTS_ROOT,
    build_coreset_module,
    build_data_module,
    build_evaluator,
    build_generator,
    build_learner,
    save_summary,
    summarize,
)

print("ROOT:", ROOT)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
# Config
METHODS = ["random", "k_centers"]
TASKS = ["ag_news", "sst2"]
LEARNER_MODELS = [
    "bert-base-uncased",
    "roberta-base",
    "albert-base-v2",
    "bert-large-uncased",
    "roberta-large",
    "microsoft/deberta-v3-base",
    "xlnet-base-cased",
]
DPC_GRID = [1, 5, 10, 20, 50, 100, 500, 1000]

SEED = 42
N_DATASET = 1
SKIP_EXISTING = True

# Same evaluation config as the herding report / previous coreset run.
EVAL_KW = dict(
    train_step=200,
    batch_size=64,
    lr=1e-4,
    n_eval_per_dataset=3,
    bf16=True,
)

SCALING_ROOT = RESULTS_ROOT / "scaling_laws"
SCALING_ROOT.mkdir(parents=True, exist_ok=True)
CORESET_CSV = SCALING_ROOT / "scaling_laws_all_methods.csv"
MISSING_GRID_CSV = RESULTS_ROOT / "scaling laws agg" / "scaling_laws_missing_grid.csv"
AGG_SCRIPT = RESULTS_ROOT / "scaling laws agg" / "aggregate_scaling_laws.py"

set_seed(SEED)
mlflow.set_tracking_uri(f"file:{RESULTS_ROOT}/mlruns")
mlflow.set_experiment("scaling_laws.missing_agnews_sst2")

print("methods:", METHODS)
print("tasks:", TASKS)
print("models:", LEARNER_MODELS)
print("dpc:", DPC_GRID)

In [ ]:
def safe_name(value: str) -> str:
    return value.replace("/", "__")


def run_dir(method: str, task: str, learner_model: str, dpc: int) -> Path:
    return SCALING_ROOT / method / task / safe_name(learner_model) / f"dpc{dpc}_seed{SEED}"


def summary_path(method: str, task: str, learner_model: str, dpc: int) -> Path:
    return run_dir(method, task, learner_model, dpc) / "summary.json"


def _model_available(name: str) -> bool:
    if not _HF_CACHE.exists():
        return True
    safe = name.replace("/", "--")
    return (_HF_CACHE / "hub" / f"models--{safe}").exists()


def _task_available(task: str) -> bool:
    if not _HF_CACHE.exists():
        return True
    if task == "ag_news":
        return (_HF_CACHE / "datasets" / "ag_news").exists() or (_HF_CACHE / "hub" / "datasets--ag_news").exists()
    return (_HF_CACHE / "datasets" / "glue" / task).exists()


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def score_from_summary(summary: dict) -> float:
    metric = summary["metric"]
    return float(summary[f"{metric}_mean"])


def coreset_row_from_summary(summary: dict) -> dict:
    return {
        "method": summary["method"],
        "task": summary["task"],
        "learner": summary["learner"],
        "dpc": int(summary["dpc"]),
        "k": int(summary["k"]),
        "score": score_from_summary(summary),
    }


def load_existing_coreset_csv() -> pd.DataFrame:
    cols = ["method", "task", "learner", "dpc", "k", "score"]
    if not CORESET_CSV.exists():
        return pd.DataFrame(columns=cols)
    df = pd.read_csv(CORESET_CSV)
    if df.empty:
        return pd.DataFrame(columns=cols)
    df = df[cols].copy()
    df = df[df["method"].isin(METHODS)]
    df["dpc"] = df["dpc"].astype(int)
    df["k"] = df["k"].astype(int)
    return df


def load_summary_rows() -> pd.DataFrame:
    rows = []
    for path in SCALING_ROOT.rglob("summary.json"):
        summary = json.loads(path.read_text())
        if summary.get("method") in METHODS and summary.get("task") in TASKS:
            rows.append(coreset_row_from_summary(summary))
    return pd.DataFrame(rows, columns=["method", "task", "learner", "dpc", "k", "score"])


def existing_keys() -> set[tuple[str, str, str, int]]:
    frames = [load_existing_coreset_csv(), load_summary_rows()]
    df = pd.concat(frames, ignore_index=True)
    if df.empty:
        return set()
    return set(zip(df["method"], df["task"], df["learner"], df["dpc"].astype(int)))


def upsert_coreset_csv(summary: dict) -> None:
    current = load_existing_coreset_csv()
    new_row = pd.DataFrame([coreset_row_from_summary(summary)])
    out = pd.concat([current, new_row], ignore_index=True)
    out["dpc"] = out["dpc"].astype(int)
    out["k"] = out["k"].astype(int)
    out = out.drop_duplicates(["method", "task", "learner", "dpc"], keep="last")
    out = out.sort_values(["method", "task", "learner", "dpc"])
    out.to_csv(CORESET_CSV, index=False)


def build_plan() -> pd.DataFrame:
    available_models = [m for m in LEARNER_MODELS if _model_available(m)]
    available_tasks = [t for t in TASKS if _task_available(t)]
    skipped_models = sorted(set(LEARNER_MODELS) - set(available_models))
    skipped_tasks = sorted(set(TASKS) - set(available_tasks))
    if skipped_models:
        print("Skip models not found in hf_cache:", skipped_models)
    if skipped_tasks:
        print("Skip tasks not found in hf_cache:", skipped_tasks)

    if MISSING_GRID_CSV.exists():
        plan = pd.read_csv(MISSING_GRID_CSV)
        plan = plan[
            plan["method"].isin(METHODS)
            & plan["task"].isin(available_tasks)
            & plan["learner"].isin(available_models)
            & plan["dpc"].isin(DPC_GRID)
        ].copy()
    else:
        plan = pd.DataFrame(
            [
                {"method": method, "task": task, "learner": learner, "dpc": dpc}
                for method in METHODS
                for task in available_tasks
                for learner in available_models
                for dpc in DPC_GRID
            ]
        )

    if SKIP_EXISTING:
        done = existing_keys()
        plan = plan[
            ~plan.apply(lambda r: (r["method"], r["task"], r["learner"], int(r["dpc"])) in done, axis=1)
        ].copy()

    plan["dpc"] = plan["dpc"].astype(int)
    plan = plan.sort_values(["method", "task", "learner", "dpc"]).reset_index(drop=True)
    return plan


plan = build_plan()
print("planned runs:", len(plan))
if not plan.empty:
    display(plan.groupby(["method", "task", "learner"]).size().rename("n_dpc").reset_index())
    display(plan.head(20))

In [ ]:
def run_group(method: str, task: str, learner_model: str, dpcs: list[int]) -> list[dict]:
    print(f"\n=== {method} | {task} | {learner_model} | dpcs={dpcs} ===")
    rows = []

    learner = build_learner(learner_model, task, gradient_checkpointing=True)
    generator = build_generator(task, gradient_checkpointing=True)
    data_module = build_data_module(task, learner, generator, train_batch_size=128)
    evaluator = build_evaluator(task, **EVAL_KW)
    selector = build_coreset_module(task, method, data_module)
    metric_key = evaluator.metric_key

    try:
        for dpc in dpcs:
            if SKIP_EXISTING and (method, task, learner_model, int(dpc)) in existing_keys():
                print(f"  skip existing: dpc={dpc}")
                continue

            k = int(dpc) * data_module.num_labels
            rd = run_dir(method, task, learner_model, int(dpc))
            rd.mkdir(parents=True, exist_ok=True)
            print(f"  run dpc={dpc}, k={k}")

            t0 = time.perf_counter()
            distilled = selector.generate_dataset(dpc=int(dpc), n=N_DATASET)
            selection_time_sec = time.perf_counter() - t0

            t1 = time.perf_counter()
            with mlflow.start_run(run_name=f"{method}.{task}.{safe_name(learner_model)}.dpc{dpc}.seed{SEED}"):
                mlflow.log_params({
                    "method": method,
                    "task": task,
                    "learner": learner_model,
                    "dpc": int(dpc),
                    "k": k,
                    "n_dataset": N_DATASET,
                    "seed": SEED,
                    **EVAL_KW,
                })
                results = evaluator.evaluate(
                    dataset_list=distilled,
                    learner=learner,
                    data_module=data_module,
                    save_result_dir=str(rd / "metrics"),
                    verbose=True,
                )
            eval_time_sec = time.perf_counter() - t1

            summary = summarize(
                results,
                metric_key,
                name=f"{method}_dpc{dpc}_seed{SEED}",
                method=method,
                task=task,
                learner=learner_model,
                dpc=int(dpc),
                k=k,
                n_dataset=N_DATASET,
                seed=SEED,
                selection_time_sec=selection_time_sec,
                eval_time_sec=eval_time_sec,
            )
            save_summary(summary, rd)
            upsert_coreset_csv(summary)
            rows.append(summary)
            print(f"  {metric_key}={summary[f'{metric_key}_mean']:.4f}  eval_time={eval_time_sec:.0f}s")
    finally:
        del learner, generator, data_module, evaluator, selector
        cleanup_cuda()

    return rows


all_rows = []
errors = []

for (method, task, learner_model), group in plan.groupby(["method", "task", "learner"]):
    dpcs = group["dpc"].astype(int).tolist()
    try:
        all_rows.extend(run_group(method, task, learner_model, dpcs))
    except Exception as exc:
        cleanup_cuda()
        err = {
            "method": method,
            "task": task,
            "learner": learner_model,
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": traceback.format_exc(),
        }
        errors.append(err)
        err_dir = SCALING_ROOT / method / task / safe_name(learner_model)
        err_dir.mkdir(parents=True, exist_ok=True)
        (err_dir / "error.json").write_text(json.dumps(err, indent=2))
        print("FAILED", method, task, learner_model, type(exc).__name__, exc)

print("new summaries:", len(all_rows))
print("errors:", len(errors))

In [ ]:
# Rebuild aggregate CSV/coverage/SVG after the new points were upserted.
if AGG_SCRIPT.exists():
    runpy.run_path(str(AGG_SCRIPT), run_name="__main__")
else:
    print("Aggregation script not found:", AGG_SCRIPT)

print("coreset csv:", CORESET_CSV)
print("aggregate csv:", RESULTS_ROOT / "scaling laws agg" / "scaling_laws_aggregated.csv")
print("aggregate plot:", RESULTS_ROOT / "scaling laws agg" / "scaling_laws_all_subplots.svg")